[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module2/02-decorators.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module2/02-decorators.ipynb)

# Decorators
**Module 2 — Intermediate Python | Estimated time: 35 minutes**

## Learning Objectives
- Understand **closures** and how they capture variables from enclosing scopes
- Implement a **function decorator** from scratch using the wrapper pattern
- Use `@syntax` sugar and preserve metadata with **`@functools.wraps`**
- Build practical decorators: `@timer`, `@retry`, `@cache`
- Create **decorators with arguments** using the factory pattern
- Write a **class-based decorator** using `__call__`
- Understand the order of **stacked decorators**

In [ ]:
import time
import functools
import traceback
from typing import Callable, Any

print('Setup complete.')

## 1. Closures — The Foundation

A **closure** is a function that captures variables from its enclosing scope.  
The inner function *closes over* the outer variable — it can still access it even after the outer function has returned.

In [ ]:
# Example 1: counter factory
def make_counter(start: int = 0):
    """Returns a function that increments and returns an internal count."""
    count = start           # `count` is captured by the closure

    def increment(step: int = 1) -> int:
        nonlocal count      # tell Python we want to rebind the outer variable
        count += step
        return count

    return increment


counter_a = make_counter()
counter_b = make_counter(100)

print(counter_a())      # 1
print(counter_a())      # 2
print(counter_b())      # 101  — independent state
print(counter_a(5))     # 7


# Example 2: multiplier factory
def multiplier(factor: float) -> Callable[[float], float]:
    """Returns a function that multiplies its argument by `factor`."""
    return lambda x: x * factor

double = multiplier(2)
triple = multiplier(3)
tax    = multiplier(1.2)

print(double(7), triple(7), tax(100))

## 2. The Decorator Pattern

A **decorator** is a callable that accepts a function, wraps it with new behaviour, and returns the wrapped version.  
This is just a closure that happens to take a *function* as its captured variable.

In [ ]:
# Step 1 — write a simple wrapper manually
def shout_wrapper(func):
    """Make any function print its result in upper case."""
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        if isinstance(result, str):
            print(result.upper())
        return result
    return wrapper


def greet(name: str) -> str:
    return f'Hello, {name}!'


# Manual application
shouting_greet = shout_wrapper(greet)
shouting_greet('world')

# Step 2 — the @ syntax is identical but cleaner
@shout_wrapper
def farewell(name: str) -> str:
    return f'Goodbye, {name}!'

farewell('world')

# @decorator is exactly equivalent to:  farewell = shout_wrapper(farewell)

## 3. Preserving Metadata with `@functools.wraps`

Without `@functools.wraps`, the wrapper replaces `__name__`, `__doc__`, etc. with its own — which breaks introspection and documentation tools.

In [ ]:
import functools

# Bad: metadata is lost
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def my_func():
    """I do something important."""
    pass

print('Without @wraps:')
print('  __name__:', my_func.__name__)   # 'wrapper' — WRONG
print('  __doc__: ', my_func.__doc__)    # None — WRONG


# Good: @functools.wraps copies the original's metadata
def good_decorator(func):
    @functools.wraps(func)               # <-- add this one line
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def my_func2():
    """I do something important."""
    pass

print('\nWith @wraps:')
print('  __name__:', my_func2.__name__)  # 'my_func2' — correct
print('  __doc__: ', my_func2.__doc__)   # docstring preserved

## 4. Practical Decorator — `@timer`

Measures how long a function takes to run and prints the elapsed time.

In [ ]:
import time
import functools

def timer(func):
    """Print the runtime of the decorated function."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f'[timer] {func.__name__!r} finished in {elapsed:.4f}s')
        return result
    return wrapper


@timer
def slow_sum(n: int) -> int:
    """Sum integers 0..n using a deliberate loop."""
    total = 0
    for i in range(n):
        total += i
    return total


result = slow_sum(5_000_000)
print(f'Result: {result:,}')

## 5. Practical Decorator — `@retry`

Automatically retries a function up to N times if it raises an exception — useful for flaky network calls.

In [ ]:
import functools
import random
import time

def retry(max_attempts: int = 3, delay: float = 0.2, exceptions=(Exception,)):
    """Decorator factory: retry `func` up to `max_attempts` times."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    last_exc = exc
                    print(f'  Attempt {attempt}/{max_attempts} failed: {exc}')
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise last_exc
        return wrapper
    return decorator


# Simulate a flaky network call
_call_count = 0

@retry(max_attempts=4, delay=0.05)
def fetch_data(url: str) -> str:
    global _call_count
    _call_count += 1
    if _call_count < 3:                          # fail on first two calls
        raise ConnectionError('Network timeout')
    return f'<html from {url}>'


html = fetch_data('https://example.com')
print('Success:', html)

## 6. Practical Decorator — `@cache`

`functools.lru_cache` memoises a function's results. We'll also build a simple manual version to understand the internals.

In [ ]:
import functools
import time

# --- Manual cache decorator ---
def simple_cache(func):
    """Cache the return value of a function based on its arguments."""
    _cache = {}                         # dict lives inside the closure
    @functools.wraps(func)
    def wrapper(*args):
        if args not in _cache:
            _cache[args] = func(*args)
        return _cache[args]
    wrapper.cache_info = lambda: _cache # expose the cache for inspection
    return wrapper

@simple_cache
def fib(n: int) -> int:
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

print('fib(35) =', fib(35))
print('Cache entries:', len(fib.cache_info()))

# --- Standard library version ---
@functools.lru_cache(maxsize=128)
def fib2(n: int) -> int:
    if n <= 1:
        return n
    return fib2(n - 1) + fib2(n - 2)

start = time.perf_counter()
print('fib2(40) =', fib2(40))
print(f'Time: {time.perf_counter() - start:.6f}s')
print(fib2.cache_info())

## 7. Class-Based Decorator with `__call__`

A class whose instances are callable (they implement `__call__`) can be used as a decorator.  
This is handy when you need to store more state than a simple closure allows.

In [ ]:
import functools
import time

class RateLimit:
    """Class-based decorator that enforces a minimum interval between calls."""

    def __init__(self, calls_per_second: float = 1.0):
        self.min_interval = 1.0 / calls_per_second
        self._last_called: float = 0.0

    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            elapsed = time.perf_counter() - self._last_called
            wait = self.min_interval - elapsed
            if wait > 0:
                print(f'  [rate-limit] waiting {wait:.2f}s...')
                time.sleep(wait)
            self._last_called = time.perf_counter()
            return func(*args, **kwargs)
        return wrapper


@RateLimit(calls_per_second=5)   # at most 5 calls per second
def ping(host: str) -> str:
    return f'pong from {host}'


for i in range(3):
    print(ping('server.local'))

## 8. Stacking Multiple Decorators

Decorators are applied **bottom-up**: the decorator closest to the `def` is applied first.

In [ ]:
import functools
import time

def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f'<b>{func(*args, **kwargs)}</b>'
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f'<i>{func(*args, **kwargs)}</i>'
    return wrapper

def uppercase(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper


# Stacking order matters:
# @bold          ← applied 3rd (outermost)
# @italic        ← applied 2nd
# @uppercase     ← applied 1st (innermost, closest to def)
@bold
@italic
@uppercase
def say(text: str) -> str:
    return text


print(say('hello'))   # <b><i>HELLO</i></b>

# Equivalent manual application:
manual = bold(italic(uppercase(say.__wrapped__.__wrapped__.__wrapped__)))
# Or think of it as: say = bold(italic(uppercase(original_say)))
print('\nDecorator stack order is bottom-up (uppercase → italic → bold)')

## Practice Exercises

**Exercise 1 — `@validate_types` Decorator**  
Write a decorator `validate_types` that inspects a function's annotations and raises `TypeError` if any argument is not an instance of its annotated type. Test it on a function `add(a: int, b: int) -> int`. (Hint: use `func.__annotations__` and the `inspect` module.)

**Exercise 2 — `@log_calls` with Arguments**  
Write a decorator factory `log_calls(logger_name='root')` that logs the function name, arguments, and return value using Python's `logging` module at `DEBUG` level whenever the decorated function is called.

**Exercise 3 — Stacking `@timer` and `@retry`**  
Using the `@timer` and `@retry` decorators defined above, decorate a function `unstable_sqrt(x)` that randomly raises `ValueError` 60% of the time and otherwise returns `math.sqrt(x)`. Apply both decorators so the total time (including retries) is measured. What order should they be stacked in?